# 02 · Customer segmentation: RFM rules vs. K-Means

Two complementary views of the same 5,833 customers (net revenue > 0, snapshot 2011-12-10):

* **RFM segments:** transparent rules on Recency and Frequency quintiles that marketing teams already know.
* **K-Means clusters:** data-driven groups on five log-scaled behavioural features.

Both are enriched with **P(alive)** from the BG/NBD model (notebook 03) to quantify revenue at risk.

In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from retail_clv import data, plots

plots.apply_style()
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 20)
tx, report = data.clean(data.load_raw(data.download()))
inv = data.invoices(tx)
metrics = json.loads(Path("reports/metrics.json").read_text())

In [2]:
seg = pd.read_csv("reports/segment_summary.csv", index_col=0)
cols = [
    "customers",
    "customer_share",
    "revenue_share",
    "median_recency",
    "median_frequency",
    "mean_p_alive",
    "revenue_last_12m",
    "revenue_at_risk",
    "action",
]
seg[cols]

,customers,customer_share,revenue_share,median_recency,median_frequency,mean_p_alive,revenue_last_12m,revenue_at_risk,action
segment,,,,,,,,,
Champions,825,0.141,0.523,8.000,12.000,0.988,"4,502,921.920","27,968.930","Reward, early access to new ranges, ask for re..."
Loyal,1148,0.197,0.278,52.000,8.000,0.958,"2,197,237.160","85,935.735","Upsell higher-value ranges, loyalty programme"
At Risk,755,0.129,0.059,374.000,4.000,0.625,"159,529.900","35,025.571",Personal win-back outreach before they churn
Potential Loyalists,712,0.122,0.038,23.000,3.000,0.940,"490,909.230","29,201.577","Membership offer, personalised recommendations"
Hibernating,1506,0.258,0.037,432.000,1.000,0.914,"136,859.420","12,099.736",Low-cost automated reactivation; suppress from...
Can't Lose,72,0.012,0.029,314.500,11.000,0.299,"46,489.090","29,186.419","High-touch win-back (account manager call), su..."
Need Attention,270,0.046,0.021,105.000,3.000,0.901,"207,984.790","17,376.041",Time-limited offers based on purchase history
About to Sleep,381,0.065,0.012,92.000,1.000,0.944,"172,511.720","11,749.291",Reactivation campaign with popular products
Promising,113,0.019,0.002,37.000,1.000,1.000,"35,277.540",0.000,"Brand awareness, free-shipping threshold nudges"


![segments](../reports/figures/rfm_segments.png)

In [3]:
r = metrics["rfm"]
print(
    f"Champions + Loyal: {r['champions_loyal_customer_share']:.0%} of customers -> {r['champions_loyal_revenue_share']:.0%} of revenue"
)
print(
    f"At Risk + Can't Lose: {r['at_risk_cant_lose_customers']:,} customers, "
    f"£{r['at_risk_cant_lose_revenue_last_12m']:,.0f} revenue in the last 12 months, "
    f"£{r['at_risk_cant_lose_revenue_at_risk']:,.0f} expected to be lost (revenue x (1 - P(alive)))"
)

Champions + Loyal: 34% of customers -> 80% of revenue
At Risk + Can't Lose: 827 customers, £206,019 revenue in the last 12 months, £64,212 expected to be lost (revenue x (1 - P(alive)))


**Reading the segments**

* **Champions and Loyal** customers are about a third of the base and generate about 80 % of net revenue. Protect them first.
* **At Risk / Can't Lose** customers used to buy often but have gone quiet. The *revenue at risk* column multiplies each customer's last-12-month revenue by their probability of having churned, which gives a direct estimate of how much a win-back campaign could save. It also sets a ceiling on what that campaign should cost.
* **Hibernating** is the largest segment by headcount but contributes little revenue. Keep these customers in cheap automated channels only.

> ⚠️ **Model caveat.** BG/NBD assumes a customer can only drop out *right after a repeat purchase*, so anyone with a single purchase gets P(alive) = 1. That is why *Hibernating* and *Promising* (mostly one-time buyers) show P(alive) ≈ 0.9–1.0. Their revenue-at-risk figures are therefore **lower bounds**. The estimates are meaningful for repeat customers such as *At Risk* (0.63) and *Can't Lose* (0.30). The MBG/NBD variant, which lets customers drop out before their first repeat purchase, would remove this artefact.

## K-Means on log-scaled behaviour

In [4]:
pd.DataFrame(metrics["kmeans"]["k_scores"]).set_index("k")

,inertia,silhouette,davies_bouldin
k,,,
2,"17,073.083",0.353,1.074
3,"13,595.642",0.266,1.245
4,"11,778.685",0.241,1.306
5,"10,500.934",0.235,1.274
6,"9,556.781",0.230,1.255
7,"8,806.984",0.219,1.259
8,"8,237.933",0.221,1.240


![k selection](../reports/figures/kmeans_k_selection.png)

**Choosing k.** The silhouette score is highest at k = 2, but two clusters only separate *active* from *lapsed* customers, which a single recency rule already does. **k = 4** keeps a reasonable silhouette (0.24, flat from k = 4 to 6) and adds a **value** dimension that is actionable. This is a deliberate trade-off: cluster metrics guide the choice, and business usefulness decides it.

In [5]:
prof = pd.read_csv("reports/cluster_profiles.csv", index_col=0)
prof[
    [
        "name",
        "customers",
        "customer_share",
        "revenue_share",
        "median_recency",
        "median_frequency",
        "median_monetary",
        "median_basket",
    ]
]

,name,customers,customer_share,revenue_share,median_recency,median_frequency,median_monetary,median_basket
cluster,,,,,,,,
0,High-value active,1188,0.204,0.756,19.000,13.000,"4,990.865",404.273
2,Mid-value active,1602,0.275,0.123,32.000,5.000,"1,107.420",246.454
3,Mid-value lapsed,1583,0.271,0.103,360.000,2.000,726.340,377.015
1,Low-value lapsed,1460,0.250,0.019,372.000,1.000,186.630,141.452


![cluster profiles](../reports/figures/cluster_profiles.png)

## Do the two views agree?

In [6]:
ct = pd.DataFrame(metrics["kmeans"]["crosstab_rfm_vs_cluster"]).fillna(0).astype(int)
ct = ct[prof["name"]]
ct.loc[ct.sum(axis=1).sort_values(ascending=False).index]

,High-value active,Mid-value active,Mid-value lapsed,Low-value lapsed
Hibernating,0,0,656,850
Loyal,518,576,28,26
Champions,583,240,1,1
At Risk,18,83,488,166
Potential Loyalists,22,532,49,109
About to Sleep,0,8,200,173
Need Attention,4,103,113,50
Promising,0,6,43,64
Can't Lose,43,25,3,1
New Customers,0,29,2,20


The views are consistent where they should be: almost all **Champions** fall into the *high-value active* or *mid-value active* clusters, and **Hibernating** customers fall into the two *lapsed* clusters. K-Means adds one thing the RFM grid cannot show: the **mid-value lapsed** cluster has a *larger average basket* than the active mid-value group. These are lapsed wholesale-style buyers, the most valuable target for a win-back campaign.